# Analiza streszczeń literatury młodzieżowej


In [273]:
## Import bibliotek

from nltk.corpus import PlaintextCorpusReader
from nltk.tokenize import word_tokenize
import pandas as pd
import string
import numpy as np
from nltk.util import ngrams
from scipy.cluster.hierarchy import dendrogram
from sklearn.cluster import AgglomerativeClustering
from sklearn.decomposition import LatentDirichletAllocation, NMF
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
import operator as op
import itertools as it
import nltk
import morfeusz2
import json
import os
from wordcloud import WordCloud
from matplotlib import pyplot as plt


In [274]:

# Zdefiniowanie funkcji pomocniczych

def print_dict(obj):
    print(json.dumps(obj, indent=4, ensure_ascii=False))

def create_dir(path):
    if not os.path.exists(path):
        os.makedirs(path)

In [275]:
## Utwórz korpus dokumentów

corpus_dir = "./Literature"
# nltk.pathsec.ENFORCE = False
nltk.data.path.append(corpus_dir)
nltk.download('punkt_tab')
corpus = PlaintextCorpusReader(corpus_dir, r'.*\.txt')
file_names = corpus.fileids()

[nltk_data] Error loading punkt_tab: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:1020)>


In [276]:
documents_dict = {}
for file_name in file_names:
    documents_dict[file_name] = corpus.raw(file_name)
print(f"Wczytano {len(documents_dict)} dokumentów.")

lengths = {}
for file_name in file_names:
    lengths[file_name] = {"pre": len(word_tokenize(documents_dict[file_name], preserve_line=True))}

Wczytano 20 dokumentów.


In [277]:
# Utowrzenie macierzy częstości
documents_df_pre = pd.DataFrame.from_dict(documents_dict, orient='index', columns=['content'])

matrix_tf = CountVectorizer().fit_transform(documents_df_pre['content'])

sparsity_tf = (1- (matrix_tf.nnz / (matrix_tf.shape[0] * matrix_tf.shape[1]))) * 100

In [278]:
matrices = {
    "pre": {
        "words": matrix_tf.shape[1],
        "sparsity": sparsity_tf,
    }
}

In [279]:
## Wstępne przygotowanie dokumentów


print(json.dumps(lengths, indent=4, ensure_ascii=False))

stopwords_path = "./stopwords_pl.txt"
with open(stopwords_path, "r", encoding="utf-8") as stopwords_file:
    stopwords = stopwords_file.read().splitlines()
    stopwords = stopwords[4:]

stopwords


{
    "Harry Potter i Czara Ognia.txt": {
        "pre": 6315
    },
    "Harry Potter i Insygnia Smierci.txt": {
        "pre": 2790
    },
    "Harry Potter i Kamien Filozoficzny.txt": {
        "pre": 2073
    },
    "Harry Potter i Komnata Tajemnic.txt": {
        "pre": 1541
    },
    "Harry Potter i Ksiaze Polkrwi.txt": {
        "pre": 1881
    },
    "Harry Potter i Przeklete Dziecko.txt": {
        "pre": 2961
    },
    "Harry Potter i Wiezien Azkabanu.txt": {
        "pre": 2075
    },
    "Harry Potter i Zakon Feniksa.txt": {
        "pre": 1685
    },
    "Opowiesci z Narni. Kon i jego chlopiec.txt": {
        "pre": 169
    },
    "Opowiesci z Narni. Ksiaze Kaspian.txt": {
        "pre": 2223
    },
    "Opowiesci z Narni. Lew, czarownica i stara szafa.txt": {
        "pre": 2270
    },
    "Opowiesci z Narni. Ostatnia bitwa.txt": {
        "pre": 139
    },
    "Opowiesci z Narni. Podroz Wedrowca do Switu.txt": {
        "pre": 1097
    },
    "Opowiesci z Narni. Siostr

['vol',
 'o.o.',
 'mgr',
 'godz',
 'zł',
 'www',
 'pl',
 'ul',
 'tel',
 'hab',
 'prof',
 'inż',
 'dr',
 'i',
 'u',
 'aby',
 'albo',
 'ale',
 'ani',
 'aż',
 'bardzo',
 'bez',
 'bo',
 'bowiem',
 'by',
 'byli',
 'bym',
 'był',
 'była',
 'było',
 'były',
 'być',
 'będzie',
 'będą',
 'chce',
 'choć',
 'co',
 'coraz',
 'coś',
 'czy',
 'czyli',
 'często',
 'dla',
 'do',
 'gdy',
 'gdyby',
 'gdyż',
 'gdzie',
 'go',
 'ich',
 'im',
 'inne',
 'iż',
 'ja',
 'jak',
 'jakie',
 'jako',
 'je',
 'jednak',
 'jednym',
 'jedynie',
 'jego',
 'jej',
 'jest',
 'jeszcze',
 'jeśli',
 'jeżeli',
 'już',
 'ją',
 'kiedy',
 'kilku',
 'kto',
 'która',
 'które',
 'którego',
 'której',
 'który',
 'których',
 'którym',
 'którzy',
 'lat',
 'lecz',
 'lub',
 'ma',
 'mają',
 'mamy',
 'mi',
 'miał',
 'mimo',
 'mnie',
 'mogą',
 'może',
 'można',
 'mu',
 'musi',
 'na',
 'nad',
 'nam',
 'nas',
 'nawet',
 'nic',
 'nich',
 'nie',
 'niej',
 'nim',
 'niż',
 'no',
 'nowe',
 'np',
 'nr',
 'o',
 'od',
 'ok',
 'on',
 'one',
 'oraz',
 '

In [280]:
#use morfeusz 
def lematizacja(text):
    morfeusz = morfeusz2.Morfeusz()
    segments = it.groupby(morfeusz.analyse(text), key=op.itemgetter(0, 1))
    def disamiguate(group):
        pairs = ((len(descr), lemma) for _, _, (_, lemma, descr, _, _, ) in group)
        perp1, lemma = min(pairs)
        return lemma.split(":")[0] if lemma else None

    lemmas = [disamiguate(group) for _, group in segments]
    return " ".join(filter(str.isalpha, lemmas))

In [281]:
for file_name in file_names:
    documents_dict[file_name] = documents_dict[file_name].lower()
    documents_dict[file_name] = "".join([char for char in documents_dict[file_name] if char not in string.punctuation])
    documents_dict[file_name] = lematizacja(documents_dict[file_name])
    documents_dict[file_name] = " ".join([word for word in word_tokenize(documents_dict[file_name], language='polish', preserve_line=True) if word not in stopwords])

for file_name in documents_dict:
    lengths[file_name]["post"] = len(word_tokenize(documents_dict[file_name], preserve_line=True))

lengths = pd.DataFrame.from_dict(lengths, orient='index')
lengths["change"] = lengths["post"] - lengths["pre"]
lengths["change_percent"] = lengths["change"] / lengths["pre"] * 100
display(lengths)

,pre,post,change,change_percent
Harry Potter i Czara Ognia.txt,6315,3806,-2509,-39.730800
Harry Potter i Insygnia Smierci.txt,2790,1681,-1109,-39.749104
Harry Potter i Kamien Filozoficzny.txt,2073,1234,-839,-40.472745
Harry Potter i Komnata Tajemnic.txt,1541,888,-653,-42.375081
Harry Potter i Ksiaze Polkrwi.txt,1881,1141,-740,-39.340776
Harry Potter i Przeklete Dziecko.txt,2961,1742,-1219,-41.168524
Harry Potter i Wiezien Azkabanu.txt,2075,1241,-834,-40.192771
Harry Potter i Zakon Feniksa.txt,1685,1039,-646,-38.338279
Opowiesci z Narni. Kon i jego chlopiec.txt,169,106,-63,-37.278107
Opowiesci z Narni. Ksiaze Kaspian.txt,2223,1299,-924,-41.565452


In [282]:
documents_df_post = pd.DataFrame.from_dict(documents_dict, orient='index', columns=['content'])

cv = CountVectorizer()
matrix_tf = cv.fit_transform(documents_df_post['content'])
tv = TfidfVectorizer()
matrix_tfidf = tv.fit_transform(documents_df_post['content'])

sparsity_tf = (1- (matrix_tf.nnz / (matrix_tf.shape[0] * matrix_tf.shape[1]))) * 100

matrices["post"] = {
    "words": matrix_tf.shape[1],
    "sparsity": sparsity_tf,
}

matrices = pd.DataFrame.from_dict(matrices, orient='index')
display(matrices)

,words,sparsity
pre,7469,90.955951
post,4675,89.312299


In [283]:
# Przygotowanie katalogów wyjściowych

create_dir("./wordclouds")
create_dir("./wordclouds/pre")
create_dir("./wordclouds/post")
create_dir("./topic_models")
create_dir("./topic_models/topics")
create_dir("./topic_models/documents")
create_dir("./clustering")
create_dir("./ngrams")



In [284]:
# Wordclouds

wordcloud = WordCloud(
    background_color="white",
    max_words=5000,
    contour_color="steelblue",
    contour_width=3
)

for index, row in documents_df_pre.iterrows():
    wordcloud.generate(row['content'])
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis("off")
    plt.title(index.replace(".txt", ""))
    plt.savefig(f"./wordclouds/pre/{index.replace('.txt', '.png')}")
    plt.close()
    
for index, row in documents_df_post.iterrows():
    wordcloud.generate(row['content'])
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis("off")
    plt.title(index.replace(".txt", ""))
    plt.savefig(f"./wordclouds/post/{index.replace('.txt', '.png')}")
    plt.close()

In [285]:
## Analiza tematyk


def plot_top_words(model, feature_names, n_top_words, title, subplots, colors):

    fig, axes = plt.subplots(*subplots, figsize=(30, 15), sharex=True)
    axes = axes.flatten()
    for topic_idx, topic in enumerate(model.components_):
        top_features_ind = topic.argsort()[-n_top_words:]
        top_features = feature_names[top_features_ind]
        weights = topic[top_features_ind]

        ax = axes[topic_idx]
        ax.barh(
            top_features, weights, height=0.7, color=colors[topic_idx % len(colors)]
        )
        ax.set_title(f"Topic {topic_idx + 1}", fontdict={"fontsize": 30})
        ax.tick_params(axis="both", which="major", labelsize=20)
        for i in "top right left".split():
            ax.spines[i].set_visible(False)
        fig.suptitle(title, fontsize=40)

    plt.subplots_adjust(top=0.90, bottom=0.05, wspace=0.90, hspace=0.3)
    plt.savefig(
        f"./topic_models/"
        + str(n_components)
        + "/topics/"
        + title.replace(" ", "_")
        + ".png"
    )
    plt.close()


def plot_documents(model, matrix, n_components, title, colors):
    document_topics = pd.DataFrame(model.transform(matrix))
    document_topics.columns = [f"Temat {i + 1}" for i in range(n_components)]
    document_topics.index = [
        file_name.replace(".txt", "") for file_name in documents_df_post.index
    ]
    plt.figure(figsize=(7, 4))
    left = [0] * len(document_topics)
    for i, col in enumerate(document_topics.columns):
        plt.bar(
            document_topics.index,
            document_topics[col],
            bottom=left,
            color=colors[i % len(colors)],
            label=col,
        )
        left = [
            left[j] + document_topics[col].iloc[j] for j in range(len(document_topics))
        ]
    plt.savefig(
        f"./topic_models/"
        + str(n_components)
        + "/documents/"
        + title.replace(" ", "_")
        + ".png"
    )
    plt.close()

In [286]:
colors = ["paleturquoise", "palevioletred", "mediumpurple", "springgreen", "lightcoral", "palegoldenrod", "salmon", "khaki", "skyblue", "slategray"]
feature_names = cv.get_feature_names_out()
n_top_features = 20
size = [1,4]
n_components = size[0]*size[1]

In [287]:
create_dir("./topic_models/" + str(n_components))
create_dir("./topic_models/" + str(n_components) + "/topics")
create_dir("./topic_models/" + str(n_components) + "/documents")

In [288]:
lda = LatentDirichletAllocation(
    n_components=n_components,
    max_iter=5,
    learning_method="online",
    learning_offset=50,
    random_state=0
)
lda.fit(matrix_tf)
plot_top_words(lda,feature_names, n_top_features, "Tematu w modeli LDA", size , colors)
plot_documents(lda,matrix_tf,n_components,"Tematu w modeli LDA", colors)

In [289]:
nmf_fn = NMF(
    n_components=n_components, 
    random_state=1,
    alpha_H=.00005,
    alpha_W=.0005,
    l1_ratio=.5,
)
nmf_fn.fit(matrix_tf)
plot_top_words(nmf_fn, feature_names, n_top_features, "Tematy w modeli NMF FN", size, colors)
plot_documents(nmf_fn, matrix_tf, n_components, "Dokumenty w modeli NMF FN", colors)



/Users/jakub/Studia/wojcik/TextMining/.venv/lib/python3.13/site-packages/sklearn/decomposition/_nmf.py:1723: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(


In [290]:
nmf_kl = NMF(
    n_components=n_components,
    random_state=1,
    beta_loss='kullback-leibler',
    solver='mu',
    alpha_H=0.00005,
    alpha_W=0.0005,
    l1_ratio=0.5,
)
nmf_kl.fit(matrix_tf)
plot_top_words(nmf_fn, feature_names, n_top_features, "Tematy w modeli NMF KL", size, colors)
plot_documents(nmf_fn, matrix_tf, n_components, "Dokumenty w modeli NMF KL", colors)